# Class 13 Notebook 1: Classification with k nearest neighbors (kNN)

This notebook contains several new ideas for the course.

- the kNN algorithm itself
- moving beyond pandas data frames, to arrays of numbers ready  for the algorithm
- splitting data into test and training datasets for "cross-validation"

These new ideas are going to be similar for all the machine learning algorithms that we use, so will start to feel familiar by the end of the week.

### 1. kNN Background

#### 1.1 Type of algorithm

Today, we learned that k Nearest Neighbours (kNN) is a supervised machine learning method.
It can be used for both classification and regression problems. This means that we are given a labelled dataset consisting of training observations $(x,y)$ and would like to capture the relationship between the features $x$ and the labelled outcomes $y$. 

Here, we will use kNN for classification. This means that the outcomes $y$ takes categorical values or labels, and the task then is to predict the category $y$ for new observations of $x$. 

#### 1.2 Distance metric

In the classification setting, the k-nearest neighbor algorithm essentially classifies a given “unseen” observation by a majority vote between the k most similar observations in the training set.

For this algorithm to work, it needs to calculate which observations are "most similar". Calculating similarity is the same thing as defining to a "distance metric" between any two observations that tracks how close or far apart they are.

The k-nearest-neighbor classifier commonly uses the Euclidean distance. This is the same idea as distance in the regular 3 dimensions of the real world, but extended to datasets that can have many more than 3 dimensions or features.

Statisticians and mathematicians like to describe observations, i.e. data points, using both superscript and subscript. 
We'll briefly present this.

Let $x^{(i)}$ be an observation. This is like a row in your tidy data frame.
Each observation $x^{(i)}$ has $p$ features $(x^{(i)}_{1}, x^{(i)}_{2},..., x^{(i)}_{p})$. These are like the values in  the columns of your tidy data frame, for this specific row.

The Euclidean distance between $x^{(i)}$ and $x^{(j)}$ is defined as: 

$$d(x^{(i)}, x^{(j)}) = \sqrt{(x^{(i)}_{1} - x^{(j)}_{1})^2 + (x^{(i)}_{2} - x^{(j)}_{2})^2 + ... + (x^{(i)}_{p} - x^{(j)}_{p})^2}$$

This distance is like Pythagoras rule for the length of the sides of a triangle, but extended to more than 2 dimensions. The idea is explained more in the [wikipedia article on Euclidean distance](https://en.wikipedia.org/wiki/Euclidean_distance).

Sometimes other distance metrics can be more suitable for a given setting, for example the [Manhattan or Taxicab distance](https://en.wikipedia.org/wiki/Taxicab_geometry) just adds the distances in each dimension.
The [Hamming distance](https://en.wikipedia.org/wiki/Hamming_distance) which can be extended to sequence and string data like DNA sequences, along with the [Levenshtein distance](https://en.wikipedia.org/wiki/Levenshtein_distance),  which were used to classify bacteria from sequence data in the first year course Biology 1C: Discovery.

For today, we'll use the Euclidean distance, although the kNN algorithm can work with any distance metric.

#### 1.3 Algorithm steps

The kNN algorithm consists of four steps:

    (1) Choose the number of neighbours to use, k.

    (2) Take the k nearest neighbours of the new data point, according to your distance metric.

    (3) Among these k neighbours, count the number of data points to each category.

    (4) Assign the new data point to the category with the highest count among neighbours of the new data point.


We'll apply kNN to a dataset that we've previously used on the course.

### 2. Load data


#### 2.1 Import libraries

Let's load the Python modules we will use throughout the notebook. These should be familiar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp

%matplotlib inline

#### 2.2 Read the file

We will analyse the Iris data set you have already familiarised yourself with earlier during the course. It includes 3 iris species with 50 observations each. For each observation, we have 4 features: sepal length and width as well as petal length and width, all in cm. 

**Here, the goal is to predict species from the numeric features using kNN.** So, species is the label, a categorical outcome that we aim to predict.

First, as always, we will load the data, inspect it, and plot it. In this notebook, we'll think about the plots differently in light of the task: **which numeric features of the data can help us to predict species?**

__TASK:__ Use ```pd.read_csv()``` to load the Iris data set in ```iris.csv```, in the Datasets folder.

In [ ]:
# Importing the dataset
iris = ...

#### 2.3 Summarize the Dataset

At the start, it is always best to get a quick glimpse of the data and its format. We can get a quick idea of how many samples (rows) and how many attributes (columns) the data contains using ```shape()```.

Using ```head(n)``` we can see the first $n$ lines of the data table.

Using describe(), we can get some summary statistics of the data. 

Let’s now take a look at the number of instances (rows) that belong to each Species, by using `groupby` and `size` to get the number of observations

We can view this as an absolute count.

In [ ]:

iris.groupby(EDITME).size()


### 3. Visualise the data

In __§2.3__, we took an initial look at the data format. Equally important is looking at our data by means of different visualisations to get a feel of the data before starting any further model analysis via machine learning.

__TASK__: Think about which plots you find most helpful and why!

#### 3.1. Pairplot

In multivariate analysis we are concerned with the relations between various data variables. Feature variables may be statistically independent or display statistical dependence. To get an idea, it often helps to look at these dependencies beforehand, but visualisation in high dimensions are tricky, and 2- or at most 3-dimensional visualisations are more intuitive to us. 

Looking at pairwise dependencies is useful when you want to visualize the relationship between multiple variables separately within subsets of your dataset.

__TASK:__ Use the ```pairplot()``` function from seaborn to make pairwise scatterplots, grouped by 'Species', along with the 1-d marginal distributions of features.

### 3.2. Visualizing 1-d distributions of the features

#### 3.2.1 Boxplot

Boxplots are a simple way to visualise 1-d distributions. 

__TASK:__ Make boxplots of all 4 features, grouped by species. 

In [ ]:
# Tip: The method/function ```boxplot()``` can be applied directly to your data.
iris.boxplot(...)


In [1]:
# More familiar, we can plot boxplots as subplots

# box plots in subplots
plt.figure(figsize=(10,10))
plt.subplot(2,2,1)
sns.boxplot(x='Species', y='petal_length', data=iris)
plt.subplot(2,2,2)
sns.boxplot(x='Species', y='petal_width', data=iris)
plt.subplot(2,2,3)
sns.boxplot(x='Species', y='sepal_length', data=iris)
plt.subplot(2,2,4)
sns.boxplot(x='Species', y='petal_width', data=iris)
plt.show()

NameError: name 'plt' is not defined

Box plots are not everyone's favourite as they only show summary statistics and not individual data points. We can use __strip plots__ or __violin plots__ as an alternative.

Which of the three do you prefer and why?

#### 3.2.2. Strip plots

__TASK:__ Make a figure with four subplots, for each feature, and use the ```stripplot``` function from seaborn, grouping by species. Using the ```size``` property you can control the size of dots in the plots, which can be helpful to better see the data points. 

In [2]:
# strip plots of the data
plt.figure(figsize=(10,10))
plt.subplot(2,2,1)
sns.stripplot(x='Species', y='petal_length', data=iris, size=3)
# try adding other sub-plots here
# use the pattern from the boxplots code above
plt.show()

NameError: name 'plt' is not defined

#### 3.2.3. Violin plots

__TASK:__ Make a figure with four subplots, for each feature, and use the ```violinplot``` function from seaborn, grouping by species. 

In [ ]:
# remove for students
# violin plots in subplots
plt.figure(figsize=(10,10))
plt.subplot(2,2,1)
sns.violinplot(x='Species', y='petal_length', data=iris)
# try it! you know what to do

plt.show()

Again: which plots are most informative about the data and why?

What do you conclude about the data - which of the features will be most useful to classify by species?

## 4 Preparing the data for the kNN calculation

**New idea in this course: At this point, we need to move beyond pandas dataframes.** That's because the mathematical operations involved in machine learning work with arrays and matrices that contain only numbers. This contrasts with data frames that can contain many different data types.

The numpy "Numerical Python" module does arrays, vectors, and matrices in Python and partners with scipy "Scientific Python" for computation with arrays and matrices.
In fact numpy underlies a lot of the features of pandas and statistical calculations that we've used so far.
Only now, with the calculations getting more complex for machine learning models, do we need to bring these gory details in and work with arrays directly in our code.

The machine learning models we will use come from the scikit-learn or `sklearn` Python module.
As described on the website https://scikit-learn.github.io/, scikit-learn is:

- Simple and efficient tools for predictive data analysis
- Accessible to everybody, and reusable in various contexts
- Built on NumPy, SciPy, and matplotlib
- Open source, commercially usable


#### 4.1 Dividing data into features and labels

The iris dataset contains six columns: 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'Species', and 'Field'. The quantitative features are described by columns 1-4. The Species column contains the relevant labels of observations. (The Field column isn't relevant to today's analysis).

Remember, the goal is to predict labels from the numeric features using kNN. 

So, first we need to split data into two arrays: `X` (features) and `y` (labels).

The pandas data frame method `.values` makes an array from a data frame, or in its own words "returns a NumPy representation of the DataFrame."
See the help page: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.values.html

In [ ]:
# we provide code here, based on the "slicing and dicing" material
feature_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris[feature_columns].values
ystring = iris['Species'].values

# Alternative way of selecting features and labels arrays:
# X = dataset.iloc[:, 1:5].values
# y = dataset.iloc[:, 5].values

It is always a good idea to check your objects! If you do this, you will find that both `X` and `ystring` are arrays as defined by numpy.

In [ ]:
X

In [ ]:
ystring

#### 4.2 Label encoding

More gory details... to do these calculations, *everything* needs to be a number.

Our species labels are categorical, encoded in character strings. The kNN function, `KNeighborsClassifier`, which we will use, does not accept string labels but does accept numbers. This is why we initially called the variable `ystring`.

Next, therefore, we need to transform these labels into numbers. The sklearn function `LabelEncoder` does this. 
Help page: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html

Applying `LabelEncoder` transforms so that Iris-setosa corresponds to 0, Iris-versicolor corresponds to 1 and Iris-virginica corresponds to 2. Check this below.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(ystring)

If you compare the values of `y` to `ystring` or the 'Species' column of the original data, you'll notice that the string label `versicolor` corresponds to 1, and so on.

In [ ]:
ystring

In [ ]:
y

#### 4.3 Splitting dataset into training set and test set

In machine learning, it is common to split datasets into training and test data sets. This is for cross-validation.

Essentially, splitting the data we can test the model performance on data that is independent from the training data. The training data is used to 'train' our machine learning model, i.e. to adjust the method's parameters, and the test dataset is then used to assess the performance of the model on previously unseen data using an appropriate metric of performance. 

Let's split dataset into training set and test set, to check later on whether or not our classifier works correctly.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

Check these variables that you have just created! You should find that 20% of the data is in the test set, because of the parameter `test_size = 0.2` used above.

In [ ]:
len(y)

In [ ]:
len(y_test)

*Note: Often, different features can take values on very different scales, in which case it would make sense to scale or 'normalise' data before further analysis. In our case, the different features, sepal and petal lengths and widths, take similar values, and we therefore don't have to worry about normalisation. Just keep in mind that in other data scenarios it can be important to apply feature scaling before running classification algorithms.*

### 5. Using kNN for classification

We have familiarised ourselves with the data, and prepared the data. Now let's use the KNN algorithm to predict species labels from the sepal and petal features. 

#### 5.1. Making predictions

__TASK:__ Use ```KNeighborsClassifier()``` from ```sklearn.neighbors``` to first construct a KNN classifier object and then use ```fit()``` on the object to build the classifier from the training set ```X_train```. Finally use ```predict()``` on the classifier object to make predictions for the test dataset ```X_test```.

We'll provide code for this.

In [ ]:
# Fitting classifier to the Training set
# Loading functions from scikit-learn
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import cross_val_score

# Instantiate learning model (k = 3)
classifier = KNeighborsClassifier(n_neighbors=3)

# Fitting the model
classifier.fit(X_train, y_train)

# Predicting the Test set results
y_pred = classifier.predict(X_test)

#### 5.2. Evaluating predictions

As introduced in the lecture, we can evaluate the performance of a supervised learning algorithm by assessing which labels are assigned correctly and incorrectly, using a "confusion matrix". [Wikipedia on confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix).

[Help page on `confusion_matrix` function](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)

__TASK:__ Build the confusion matrix using the scikit-learn function ```confusion_matrix()```, applied to the test labels and predicted labels. View the confusion matrix.

In [ ]:
# try this yourself
cm = ...
cm

Calculating model accuracy using ```accuracy_score()```:

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print('\n Accuracy of our model is ' + str(round(accuracy*100, 2)) + '%.')

__TASK:__ Reflect on the above result! Are you happy with this accuracy? Is accuracy indeed the best metric to use here, and if so, why? Repeat __§5.1__ & __§5.3__ with a different $k$ - when do you get better or worse predictions? Do you think the estimated accuracy is reliable?

Some blank code chunks below to try with different k - try changing the `n_neighbors` argument.

*Note: Remember to give your Python variables sensible names that avoid adding confusion. For example, if you wanted to remember which classifier used 3 neighbours and which used 4, you could call them `classifier_k3` and `classifier_k4`.*

#### 5.4. Using cross-validation for parameter tuning:

Cross-validation is a technique to systematically split up the training and test sets in different ways so as to obtain more robust performance estimates. Let's try to use it to determine the best value of $k$.

The key idea is to train the model on only some data - the training data - then score this model on different data - the test data.
The cross-validation score is essentially the accuracy score above, but calculated on the training data.

There are many different ways to split up the data into training and test, and cross validation score will be  different for each one.
This is an example of random sampling, and because the samples are taken repeatedly from the same underlying data set it is random "resampling".

__TASK:__ For a list of values $k\in[1, ..., 50]$ build a KNN classifier using again ```KNeighborsClassifier``` and then use the function ```cross_val_score()``` on ```X_train``` and ```y_train``` to obtain robust estimates of the accuracy of the classifiers with different $k$. Store the results in a list ```cv_scores```.

Here we'll use a for loop to do the calculation repeatedly for many values of k. For loops were used in some of the first year Python material, but haven't appeared much in Biology 2A. We provide most of the code.

In [ ]:
# creating list of k values for kNN
k_list = list(range(1,50,2))
# creating list of cross validation scores
cv_scores = []

# for each value of k, perform 10-fold cross validation
for k in k_list:
    knn = # EDIT TO CALCULATE kNN FOR THIS k
    scores = cross_val_score(knn, X_train, y_train, cv=10, scoring='accuracy')
    cv_scores.append(scores.mean())

__TASK:__ Let's plot the cross-validation accuracy scores against k.

In [4]:
# plot the cv_scores
sns.lineplot(x = EDITME, y = EDITMETOO)
# plot a horizontal line at 1 as placeholder for "perfect accuracy"
plt.hlines(y = 1, xmin = 0, xmax = 50, color = 'grey', linestyles = 'dashed')
plt.xlabel('Number of Neighbors k', fontsize=15)
plt.ylabel('Cross-validation scores', fontsize=15)

plt.show()

NameError: name 'sns' is not defined

__REFLECT:__ The best $k$-NN classifier is the one that maximizes the accuracy, i.e. minimises the misclassification error. Find out what the $k$ is. Now look at the trend of the accuracy - would you have expected that the error goes up again for higher $k$, and if so why?

The decline in accuracy at large k is an example of 'overfitting', which is fitting one sample dataset so well that performance on on out-of-sample data degrades.
See the [Wikipedia article on overfitting](https://en.wikipedia.org/wiki/Overfitting).
How can you detect overfitting from the above curve?

In [ ]:
# finding best k
best_k = k_list[cv_scores.index(max(cv_scores))]
print("The optimal number of neighbors is %d." % best_k)

__TASK:__ Above we have applied 10-fold cross-validation. Repeat the cross-validation analysis at different fold-values, changing the ```cv``` property in the ```cross_val_score()``` function. What changes do you see? By searching 'cross-validation' what insights can you get on how to choose the fold-value?

*Congratulations on getting to this point and learning some new ideas and machine-learning style coding.*

The kNN notebook continues with an optional section on writing your own kNN, and then a bibliography. In class 13, it's important to do the multiple regression homework notebook as well as the kNN classification homework.

You may wish to return to this notebook as homework, after working through the multiple regression notebook.

### 6. Optional: write your own KNN implementation


__Advanced Python task for the curious-minded:__ Do you think you can implement your own KNN algorithm?

We've shared one successful implementation in the answers notebook.

__Optional task:__ Start by building defining ```class MyKNeighborsClassifier()```, it should include functions ```__init__(self, params)```, ```fit(self, X, y)``` and ```predict(self, X_test)```. The ```predict()``` function will loop over all new data points and make single predictions for each point. For that, implement a separate function ```single_prediction(X, y, x_train, k)```, which computes the distance of all training data points to the new point and considers the class label of the $k$ nearest neighbours for a vote on the predicted class label. 

In [ ]:
import numpy as np
import pandas as pd

class MyKNeighborsClassifier():
    """
    My implementation of KNN algorithm.
    """
    
    def __init__(self, n_neighbors=5):
        """
        track n_neighbours
        """
        # add your code here
        
    def fit(self, X, y):
        """
        Fit the model using X as array of features and y as array of labels.
        """
        # add your code here
        
    def predict(self, X_test):
        
        # add your code here that calls single_predictions
        return(predictions)

In [ ]:
def single_prediction(X, y, x_test, k):
    # write code here to do a single prediction 
    
    return(MAXIMUM VALUE)

Now instantiate and fit your kNN model on the training set and make predictions for ```X_test``` for a $k$ of your choice.

In [ ]:
# Instantiate learning model (k = 3)
my_classifier = MyKNeighborsClassifier(n_neighbors=3)

# Fitting the model
my_classifier.fit(X_train, y_train)

# Predicting the Test set results
my_y_pred = my_classifier.predict(X_test)

__TASK:__ Compute the accuracy of your kNN classifier and compare to the result above using ```KNeighborsClassifier()```.

In [ ]:
accuracy = accuracy_score(y_test, my_y_pred) * 100
print('Accuracy of our model is equal to ' + str(round(accuracy, 2)) + ' %.')

### 7. Bibliography

This notebook has been adapted from https://www.kaggle.com/code/skalskip/iris-data-visualization-and-knn-classification. 

Some useful links:

1. MIT Lecture on kNN: https://www.youtube.com/watch?v=09mb78oiPkA
2. Theory of kNN: http://www.scholarpedia.org/article/K-nearest_neighbor
3. https://machinelearningmastery.com/tutorial-to-implement-k-nearest-neighbors-in-python-from-scratch/
4. https://kevinzakka.github.io/2016/07/13/k-nearest-neighbor/
5. https://www.analyticsvidhya.com/blog/2014/10/introduction-k-neighbours-algorithm-clustering/

*Note: remember the Class 13 homework, multiple regression notebook.*